In [ ]:
import os
import re
import csv
import random
import shutil
from datetime import datetime
import torch
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(2137)

In [ ]:
RQ1_ROOT = "../datasets/rq1_datasets"
EVENTA_ROOT = "../datasets/eventA"

RQ1_SOURCES = {
    "factory_method": {
        "true":  "../datasets/Factory method/name",
        "false": "../datasets/Factory method/nonfactorymethod",
    },
    "prototype": {
        "true":  "../datasets/Prototype/name",
        "false": "../datasets/Prototype/nonprototype",
    },
    "singleton": {
        "true":  "../datasets/Singleton/name",
        "false": "../datasets/Singleton/nonsingleton",
    },
}

def _natural_key(name):
    """Sort 'foo (2).java' before 'foo (10).java'."""
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", name)]

def build_eventA_dataset(pattern, n_total=15, seed=None):
    """Generate Event A using only files excluded from RQ1."""

    if pattern not in RQ1_SOURCES:
        raise ValueError(f"Unknown pattern: {pattern}")

    # Original source folders
    src_true = RQ1_SOURCES[pattern]["true"]
    src_false = RQ1_SOURCES[pattern]["false"]

    # Files already used in RQ1
    rq1_path = os.path.join(RQ1_ROOT, pattern)

    rq1_true = set(os.listdir(os.path.join(rq1_path, "pattern")))
    rq1_false = set(os.listdir(os.path.join(rq1_path, "non-pattern")))

    # Only use files NOT in RQ1
    true_files = sorted(
        (
            f for f in os.listdir(src_true)
            if f.endswith(".java") and f not in rq1_true
        ),
        key=_natural_key
    )

    false_files = sorted(
        (
            f for f in os.listdir(src_false)
            if f.endswith(".java") and f not in rq1_false
        ),
        key=_natural_key
    )

    # Target 7 positive + 8 negative for 15 samples
    n_true = min(n_total // 2, len(true_files))
    n_false = min(n_total - n_total // 2, len(false_files))

    # Fill shortages from either available class
    remaining = n_total - n_true - n_false

    extra_true = min(remaining, len(true_files) - n_true)
    n_true += extra_true
    remaining -= extra_true

    extra_false = min(remaining, len(false_files) - n_false)
    n_false += extra_false
    remaining -= extra_false

    if remaining:
        print(f"Warning: {pattern} is short {remaining} samples.")

    # Select files
    if seed is None:
        sel_true = true_files[:n_true]
        sel_false = false_files[:n_false]
        rule = "natural sorted order"
    else:
        rng = random.Random(seed)
        sel_true = rng.sample(true_files, n_true)
        sel_false = rng.sample(false_files, n_false)
        rule = f"random seed={seed}"

    # Create output
    target = os.path.join(EVENTA_ROOT, pattern)
    dst_true = os.path.join(target, "pattern")
    dst_false = os.path.join(target, "non-pattern")

    if os.path.exists(target):
        shutil.rmtree(target)

    os.makedirs(dst_true)
    os.makedirs(dst_false)

    for f in sel_true:
        shutil.copy2(
            os.path.join(src_true, f),
            os.path.join(dst_true, f)
        )

    for f in sel_false:
        shutil.copy2(
            os.path.join(src_false, f),
            os.path.join(dst_false, f)
        )

    # Record selection
    with open(
        os.path.join(target, "selection.csv"),
        "w",
        newline="",
        encoding="utf-8"
    ) as fh:
        writer = csv.writer(fh)
        writer.writerow(["class", "label", "filename", "source_folder"])

        writer.writerows(
            [["pattern", 1, f, src_true] for f in sel_true]
        )
        writer.writerows(
            [["non-pattern", 0, f, src_false] for f in sel_false]
        )

        writer.writerow([])
        writer.writerow([f"selection rule: {rule}"])

    print(
        f"{pattern}: {n_true} true + {n_false} false "
        f"= {n_true + n_false} files"
    )

    return {
        "pattern": pattern,
        "n_true": n_true,
        "n_false": n_false,
        "folder": target
    }


def build_all_eventA(n_total=15, seed=None):
    results = [
        build_eventA_dataset(p, n_total=n_total, seed=seed)
        for p in RQ1_SOURCES
    ]

    return results


In [ ]:
build_all_eventA(n_total=15)

In [ ]:
# Define the path to the folder containing your Java programs
rq1_programs_folder = "../datasets/rq1_datasets/factory_method" 
eventA_programs_folder = "../datasets/eventA/factory_method"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []

eventA_programs = []
eventA_labels = []
eventA_file_names = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(rq1_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)

    folder_path = os.path.join(eventA_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                eventA_programs.append(f.read())

            eventA_labels.append(label)
            eventA_file_names.append(program_file)


# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

eventA_embeddings = []
for program in tqdm(eventA_programs, desc="Generating A embeds"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    eventA_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))
eventA_norm = scaler.transform(np.vstack(eventA_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    normalized_embeddings,
    predicted_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)
A_predictions = knn.predict(eventA_norm)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

precision_A = precision_score(eventA_labels, A_predictions, average='weighted')
recall_A = recall_score(eventA_labels, A_predictions, average='weighted')
f1_A = f1_score(eventA_labels, A_predictions, average='weighted')
# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")

for filename, prediction, true_label in zip(eventA_file_names, A_predictions, eventA_labels):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision_A}, Recall: {recall_A}, F-score: {f1_A}")


In [ ]:
# Define the path to the folder containing your Java programs
rq1_programs_folder = "../datasets/rq1_datasets/prototype" 
eventA_programs_folder = "../datasets/eventA/prototype"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []

eventA_programs = []
eventA_labels = []
eventA_file_names = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(rq1_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)

    folder_path = os.path.join(eventA_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                eventA_programs.append(f.read())

            eventA_labels.append(label)
            eventA_file_names.append(program_file)


# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

eventA_embeddings = []
for program in tqdm(eventA_programs, desc="Generating A embeds"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    eventA_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))
eventA_norm = scaler.transform(np.vstack(eventA_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    normalized_embeddings,
    predicted_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)
A_predictions = knn.predict(eventA_norm)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

precision_A = precision_score(eventA_labels, A_predictions, average='weighted')
recall_A = recall_score(eventA_labels, A_predictions, average='weighted')
f1_A = f1_score(eventA_labels, A_predictions, average='weighted')
# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")

for filename, prediction, true_label in zip(eventA_file_names, A_predictions, eventA_labels):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision_A}, Recall: {recall_A}, F-score: {f1_A}")


In [ ]:
# Define the path to the folder containing your Java programs
rq1_programs_folder = "../datasets/rq1_datasets/singleton" 
eventA_programs_folder = "../datasets/eventA/singleton"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []

eventA_programs = []
eventA_labels = []
eventA_file_names = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(rq1_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)

    folder_path = os.path.join(eventA_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                eventA_programs.append(f.read())

            eventA_labels.append(label)
            eventA_file_names.append(program_file)


# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

eventA_embeddings = []
for program in tqdm(eventA_programs, desc="Generating A embeds"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    eventA_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))
eventA_norm = scaler.transform(np.vstack(eventA_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    normalized_embeddings,
    predicted_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)
A_predictions = knn.predict(eventA_norm)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

precision_A = precision_score(eventA_labels, A_predictions, average='weighted')
recall_A = recall_score(eventA_labels, A_predictions, average='weighted')
f1_A = f1_score(eventA_labels, A_predictions, average='weighted')
# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")

for filename, prediction, true_label in zip(eventA_file_names, A_predictions, eventA_labels):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision_A}, Recall: {recall_A}, F-score: {f1_A}")
